# ComfyUI (GGUF) ? FLUX.1 SRPO (Colab)

This notebook:
- installs ComfyUI + ComfyUI-Manager + ComfyUI-GGUF
- downloads FLUX.1 SRPO GGUF, text encoders, and official VAE
- launches ComfyUI via Cloudflare Tunnel

Quick start:
1. Run all cells from top to bottom.
2. Provide HF token when prompted (recommended for official gated files).
3. Open the tunnel URL and load your workflow.


## 1) Installation
Install ComfyUI, ComfyUI-Manager, ComfyUI-GGUF, and swap for better Colab stability.


In [ ]:
# @title 1) Install ComfyUI + Manager + ComfyUI-GGUF (with swap)
import os

# Small protection against RAM-related crashes
if not os.path.exists('/swapfile'):
    print('Creating swap (8GB)...')
    !sudo fallocate -l 8G /swapfile
    !sudo chmod 600 /swapfile
    !sudo mkswap /swapfile
    !sudo swapon /swapfile
    print('Swap enabled.')

# Useful packages
!apt-get -y update -qq
!apt-get -y install -qq aria2 ffmpeg

# More predictable CUDA allocator (sometimes helps reduce fragmentation)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'

# ComfyUI
if not os.path.exists('/content/ComfyUI'):
    print('Cloning ComfyUI...')
    !git clone https://github.com/comfyanonymous/ComfyUI /content/ComfyUI

%cd /content/ComfyUI

print('Installing python requirements...')
!pip install -U pip
!pip install -r requirements.txt

# Attention acceleration pack (T4-safe)
ENABLE_ACCEL_PACK = True  # Set False to skip optional attention accelerators.

if ENABLE_ACCEL_PACK:
    import sys
    import subprocess

    def pip_try(spec):
        print(f"Installing optional accelerator: {spec}")
        return subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', spec], check=False).returncode == 0

    import torch
    gpu_name = 'CPU'
    cc_major, cc_minor = (0, 0)
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        cc_major, cc_minor = torch.cuda.get_device_capability(0)

    print(f"GPU detected: {gpu_name} (sm{cc_major}{cc_minor})")

    x_ok = pip_try('xformers>=0.0.28.post3')
    print('xformers:', 'OK' if x_ok else 'FAILED (continuing)')

    # FlashAttention-3 is Hopper-only; FlashAttention-2 and SageAttention target Ampere+.
    if cc_major >= 8:
        fa_ok = pip_try('flash-attn>=2.7.0.post2')
        sage_ok = pip_try('sageattention>=2.1.1')
        print('flash-attn:', 'OK' if fa_ok else 'FAILED (continuing)')
        print('sageattention:', 'OK' if sage_ok else 'FAILED (continuing)')
    else:
        print('Skipping flash-attn and sageattention on this GPU: T4/Turing (sm75) uses xformers path.')
else:
    print('Attention acceleration pack disabled by user (ENABLE_ACCEL_PACK=False).')


# ComfyUI-Manager
if not os.path.exists('custom_nodes/ComfyUI-Manager'):
    !git clone https://github.com/ltdrdata/ComfyUI-Manager.git custom_nodes/ComfyUI-Manager

# ComfyUI-GGUF
if not os.path.exists('custom_nodes/ComfyUI-GGUF'):
    print('Installing ComfyUI-GGUF...')
    !git clone https://github.com/city96/ComfyUI-GGUF.git custom_nodes/ComfyUI-GGUF
    !pip install -r custom_nodes/ComfyUI-GGUF/requirements.txt

# ComfyUI-KJNodes
if not os.path.exists('custom_nodes/comfyui-kjnodes'):
    print('Installing ComfyUI-KJNodes...')
    !git clone https://github.com/kijai/ComfyUI-KJNodes.git custom_nodes/comfyui-kjnodes
if os.path.exists('custom_nodes/comfyui-kjnodes/requirements.txt'):
    !pip install -r custom_nodes/comfyui-kjnodes/requirements.txt



# Hugging Face authentication (interactive prompt)
!pip install -q huggingface_hub
from getpass import getpass
from huggingface_hub import login

hf_token = getpass('Enter Hugging Face token: ').strip()
if not hf_token:
    raise RuntimeError('Hugging Face token is required for this notebook run.')

login(token=hf_token, add_to_git_credential=False)
os.environ['HUGGINGFACE_TOKEN'] = hf_token
os.environ['HF_TOKEN'] = hf_token
print('HF auth: OK')

# Civitai authentication (for LoRA downloads)
civitai_token = getpass('Enter Civitai API token (optional, press Enter to skip): ').strip()
if civitai_token:
    os.environ['CIVITAI_API_TOKEN'] = civitai_token
    print('Civitai auth: OK')
else:
    print('Civitai token not set. Civitai downloads may fail (403).')

print('Done.')





#######################
#Choose GGUF quant defaults and review rough file-size/VRAM estimates.

# @title 2) Settings: GGUF quant selection + size/VRAM table
import math

# ---- Auto quant by VRAM ----
AUTO_QUANT_BY_VRAM = True
AUTO_QUANT_VRAM_FRACTION = 0.7
AUTO_VRAM_FALLBACK_GB = 14.0


# ---- Quant selection (editable) ----
SRPO_QUANT = 'Q2_K'  # options: Q2_K, Q3_K, Q4_K, Q5_K, Q6_K, Q8_0, BF16

# Quantized T5-XXL encoder (GGUF)
T5_QUANT = 'Q4_K_M'  # options: Q3_K_S, Q3_K_M, Q3_K_L, Q4_K_S, Q4_K_M, Q5_K_S, Q5_K_M, Q6_K, Q8_0, f16, f32

# If VRAM is limited, keep this enabled to offload text encoders to CPU/RAM
OFFLOAD_TEXT_ENCODERS = True

# ---- File-size tables (GB) from Hugging Face listings ----
SRPO_SIZES_GB = {
  'Q2_K': 4.02,
  'Q3_K': 5.37,
  'Q4_K': 6.93,
  'Q5_K': 8.42,
  'Q6_K': 9.85,
  'Q8_0': 12.71,
  'BF16': 23.81,
}

T5_SIZES_GB = {
  'Q3_K_S': 2.10,
  'Q3_K_M': 2.30,
  'Q3_K_L': 2.46,
  'Q4_K_S': 2.74,
  'Q4_K_M': 2.90,
  'Q5_K_S': 3.29,
  'Q5_K_M': 3.39,
  'Q6_K': 3.91,
  'Q8_0': 5.06,
  'f16': 9.53,
  'f32': 19.05,
}

CLIP_L_GB = 0.25
VAE_GB = 0.34  # official ae.safetensors




def detect_total_vram_gb(default=AUTO_VRAM_FALLBACK_GB):
    try:
        import torch
        if torch.cuda.is_available():
            return torch.cuda.get_device_properties(0).total_memory / (1024**3)
    except Exception as e:
        print(f"VRAM detection failed ({e}); using fallback {default:.2f} GB")
    return default


def _is_supported_quant_name(qname):
    q = str(qname).upper().strip()
    if q.startswith('UD-') or q.startswith('IQ'):
        return False
    if q in {'BF16', 'F16', 'F32'}:
        return True
    return q.startswith('Q')

def _supported_quant_items(size_dict):
    return sorted([(k, v) for k, v in size_dict.items() if _is_supported_quant_name(k)], key=lambda kv: kv[1])

def pick_best_quant(size_dict, budget_gb, overhead=1.10):
    items = _supported_quant_items(size_dict)
    if not items:
        items = sorted(size_dict.items(), key=lambda kv: kv[1])
    fitting = [(k, v) for k, v in items if (v * overhead) <= budget_gb]
    if fitting:
        return fitting[-1][0]
    return items[0][0]


def est_vram_gb(size_gb, overhead=1.10):
    return size_gb * overhead


def print_table(title, d):
    print('\n' + title)
    print('-' * len(title))
    for k, v in d.items():
        print(f"{k:16s}  file~{v:5.2f} GB   VRAM~{est_vram_gb(v):5.2f} GB")


# ---- Auto apply quant choice based on detected VRAM ----
TOTAL_VRAM_GB = detect_total_vram_gb()
AUTO_BUDGET_GB = TOTAL_VRAM_GB * AUTO_QUANT_VRAM_FRACTION
print(f"\nGPU VRAM detected: ~{TOTAL_VRAM_GB:.2f} GB | Auto budget (75%): ~{AUTO_BUDGET_GB:.2f} GB")

if AUTO_QUANT_BY_VRAM:
    if OFFLOAD_TEXT_ENCODERS:
        budget_srpo = max(AUTO_BUDGET_GB - est_vram_gb(VAE_GB), 0.01)
        SRPO_QUANT = pick_best_quant(SRPO_SIZES_GB, budget_srpo)
    else:
        best_fit = None
        best_any = None
        for srpo_k, srpo_v in _supported_quant_items(SRPO_SIZES_GB):
            for t5_k, t5_v in _supported_quant_items(T5_SIZES_GB):
                total = est_vram_gb(srpo_v) + est_vram_gb(t5_v) + est_vram_gb(CLIP_L_GB) + est_vram_gb(VAE_GB)
                score = (srpo_v, t5_v)
                if (best_any is None) or (total < best_any[0]):
                    best_any = (total, srpo_k, t5_k)
                if total <= AUTO_BUDGET_GB:
                    if (best_fit is None) or (score > best_fit[0]) or (score == best_fit[0] and total < best_fit[1]):
                        best_fit = (score, total, srpo_k, t5_k)

        if best_fit is not None:
            SRPO_QUANT = best_fit[2]
            T5_QUANT = best_fit[3]
        else:
            SRPO_QUANT = best_any[1]
            T5_QUANT = best_any[2]

    print(f"Auto quant active: SRPO_QUANT={SRPO_QUANT}, T5_QUANT={T5_QUANT}")

print_table('FLUX SRPO GGUF (UNet/DiT) quants', SRPO_SIZES_GB)
print_table('T5-v1.1-XXL encoder GGUF quants', T5_SIZES_GB)

print('\nSelected:')
print('  SRPO_QUANT            =', SRPO_QUANT)
print('  T5_QUANT              =', T5_QUANT)
print('  OFFLOAD_TEXT_ENCODERS =', OFFLOAD_TEXT_ENCODERS)

total = 0.0
total += est_vram_gb(SRPO_SIZES_GB[SRPO_QUANT])
if not OFFLOAD_TEXT_ENCODERS:
    total += est_vram_gb(T5_SIZES_GB[T5_QUANT])
    total += est_vram_gb(CLIP_L_GB)
total += est_vram_gb(VAE_GB)

print(f"\nRough GPU VRAM for weights: ~{total:.2f} GB")
print('Note: real peak VRAM also depends on resolution, frame count, batch, and scheduler.')


# ---- RAM-aware auto guard (Colab-safe) ----
AUTO_QUANT_BY_RAM = True
AUTO_QUANT_RAM_FRACTION = 0.6
AUTO_RAM_OS_RESERVE_GB = 2.0
AUTO_RAM_RUNTIME_GB = 1.2
AUTO_RAM_SPILL_FACTOR = 0.45
AUTO_RAM_GPU_RESIDENT_FACTOR = 0.10
AUTO_RAM_TEXT_OFFLOAD_FACTOR = 1.05
RAM_GUARD_ASSUME_SINGLE_MODEL_RUN = True


def detect_total_ram_gb(default=12.9):
    try:
        import psutil
        return psutil.virtual_memory().total / (1024**3)
    except Exception as e:
        print(f"RAM detection failed ({e}); using fallback {default:.2f} GB")
    return default


def _norm_tokens(name):
    return [t for t in str(name).upper().replace('-', '_').split('_') if t]


def _is_text_quant_var(qvar):
    toks = _norm_tokens(qvar)
    text_keys = {'T5', 'QWEN', 'UMT5', 'FLAN', 'CLIP', 'TEXT', 'ENCODER', 'TE'}
    return any((t in text_keys) or t.startswith('QWEN') or t.startswith('T5') for t in toks)


def _size_dict_for_quant_var(qvar, g):
    pref = qvar[:-6] if qvar.endswith('_QUANT') else qvar
    candidates = [
        f'{pref}_SIZES_GB',
        f'{pref}_SIZE_GB',
        f'{pref}_GGUF_GB',
    ]
    for c in candidates:
        if c in g and isinstance(g[c], dict):
            return c
    # fallback: first dict with matching prefix
    for k, v in g.items():
        if isinstance(v, dict) and k.endswith(('_SIZES_GB', '_GGUF_GB')) and pref in k:
            return k
    return None


def _smallest_quant_key(size_dict):
    if not size_dict:
        return None
    items = _supported_quant_items(size_dict)
    if items:
        return items[0][0]
    return sorted(size_dict.items(), key=lambda kv: kv[1])[0][0]

MIN_QUANT_FLOORS = {
    'T5': 'Q5_K_M',
    'QWEN': 'Q4_K_S',
    'SRPO': 'Q5_K',
    'KLEIN': 'Q5_K_M',
    'ZIMAGE': 'Q5_K_M',
    'CHROMA': 'Q5_K_M',
    'WAN': 'Q5_K_M',
    'LTX': 'Q5_K_M',
}

def _floor_quant_key(qvar, size_dict):
    q = str(qvar).upper()
    for family, floor in MIN_QUANT_FLOORS.items():
        if family in q and floor in size_dict:
            return floor
    return _smallest_quant_key(size_dict)

def _reduce_quant_with_floor(qvar, current, size_dict):
    if not isinstance(size_dict, dict) or not size_dict or current not in size_dict:
        return current

    floor = _floor_quant_key(qvar, size_dict)
    if floor not in size_dict:
        floor = _smallest_quant_key(size_dict)

    cur_size = float(size_dict[current])
    floor_size = float(size_dict[floor])

    if cur_size <= floor_size:
        return current

    candidates = sorted((k, float(v)) for k, v in size_dict.items())
    step_down = [k for k, s in candidates if floor_size <= s < cur_size]
    if step_down:
        return step_down[-1]
    return floor


def _estimate_ram_peak_gb(g):
    # Selected quant sizes
    quant_sizes = {}
    for k, v in list(g.items()):
        if not (isinstance(k, str) and k.endswith('_QUANT') and isinstance(v, str)):
            continue
        dname = _size_dict_for_quant_var(k, g)
        if not dname:
            continue
        d = g[dname]
        if v in d:
            quant_sizes[k] = float(d[v])

    # Fixed model components from scalar *_GB constants
    fixed_components = []
    for k, v in list(g.items()):
        if not (isinstance(k, str) and k.endswith('_GB') and isinstance(v, (int, float))):
            continue
        if k.startswith('AUTO_'):
            continue
        if any(x in k for x in ['VRAM', 'RAM', 'BUDGET', 'FALLBACK', 'FRACTION', 'RESERVE', 'RUNTIME', 'SPILL', 'RESIDENT']):
            continue
        fixed_components.append(float(v))

    offload = bool(g.get('OFFLOAD_TEXT_ENCODER', g.get('OFFLOAD_TEXT_ENCODERS', False)))

    text_vals = []
    core_vals = []
    for qvar, size in quant_sizes.items():
        if _is_text_quant_var(qvar):
            text_vals.append(float(size))
        else:
            core_vals.append(float(size))

    if RAM_GUARD_ASSUME_SINGLE_MODEL_RUN:
        text_gb = max(text_vals) if text_vals else 0.0
        core_gb = max(core_vals) if core_vals else 0.0
    else:
        text_gb = sum(text_vals)
        core_gb = sum(core_vals)

    fixed_gb = sum(fixed_components)
    resident_proxy = core_gb + fixed_gb

    text_ram = text_gb * (AUTO_RAM_TEXT_OFFLOAD_FACTOR if offload else 0.20)
    spill_ram = resident_proxy * AUTO_RAM_SPILL_FACTOR
    resident_ram = resident_proxy * AUTO_RAM_GPU_RESIDENT_FACTOR

    ram_peak = AUTO_RAM_OS_RESERVE_GB + AUTO_RAM_RUNTIME_GB + text_ram + spill_ram + resident_ram
    return ram_peak, quant_sizes, text_gb, core_gb, fixed_gb


TOTAL_RAM_GB = detect_total_ram_gb()
RAM_BUDGET_GB = TOTAL_RAM_GB * AUTO_QUANT_RAM_FRACTION
RAM_PEAK_GB, _ram_quant_sizes, _ram_text_gb, _ram_core_gb, _ram_fixed_gb = _estimate_ram_peak_gb(globals())

print(f"\nRAM detected: ~{TOTAL_RAM_GB:.2f} GB | RAM budget ({int(AUTO_QUANT_RAM_FRACTION*100)}%): ~{RAM_BUDGET_GB:.2f} GB")
print(f"Estimated RAM peak: ~{RAM_PEAK_GB:.2f} GB (text~{_ram_text_gb:.2f}, core~{_ram_core_gb:.2f}, fixed~{_ram_fixed_gb:.2f})")

if AUTO_QUANT_BY_RAM and RAM_PEAK_GB > RAM_BUDGET_GB:
    print("RAM guard: estimated peak exceeds budget. Trying safer quant fallback with quality floors...")

    # 1) shrink text quants first
    for qvar in sorted(list(globals().keys())):
        if not (qvar.endswith('_QUANT') and _is_text_quant_var(qvar)):
            continue
        dname = _size_dict_for_quant_var(qvar, globals())
        if not dname:
            continue
        d = globals()[dname]
        if isinstance(d, dict) and d:
                current = globals().get(qvar)
                reduced = _reduce_quant_with_floor(qvar, current, d)
                if reduced is not None and current != reduced:
                    print(f"  RAM guard: {qvar} {current} -> {reduced}")
                    globals()[qvar] = reduced

    RAM_PEAK_GB, _, _, _, _ = _estimate_ram_peak_gb(globals())

    # 2) if still high, shrink remaining quants
    if RAM_PEAK_GB > RAM_BUDGET_GB:
        for qvar in sorted(list(globals().keys())):
            if not qvar.endswith('_QUANT'):
                continue
            if _is_text_quant_var(qvar):
                continue
            dname = _size_dict_for_quant_var(qvar, globals())
            if not dname:
                continue
            d = globals()[dname]
            if isinstance(d, dict) and d:
                current = globals().get(qvar)
                reduced = _reduce_quant_with_floor(qvar, current, d)
                if reduced is not None and current != reduced:
                    print(f"  RAM guard: {qvar} {current} -> {reduced}")
                    globals()[qvar] = reduced

    RAM_PEAK_GB, _, _, _, _ = _estimate_ram_peak_gb(globals())
    if RAM_PEAK_GB > RAM_BUDGET_GB:
        print(f"RAM guard result: still risky (~{RAM_PEAK_GB:.2f} GB > ~{RAM_BUDGET_GB:.2f} GB).")
    else:
        print(f"RAM guard result: OK (~{RAM_PEAK_GB:.2f} GB <= ~{RAM_BUDGET_GB:.2f} GB).")
else:
    print("RAM guard: OK (current selection fits estimated RAM budget).")

Creating swap (8GB)...
Setting up swapspace version 1, size = 8 GiB (8589930496 bytes)
no label, UUID=33e9fba9-f269-4de8-a96e-7f9bc9c0e60c
swapon: /swapfile: swapon failed: Invalid argument
Swap enabled.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libc-ares2:amd64.


## 3) Model Download OPTION A : Recommended
Download the required model files (and optional components) into ComfyUI folders.


In [ ]:
# @title 3) Download models (FLUX.1 SRPO GGUF + quantized T5 GGUF + CLIP-L + official VAE)
import os
!wget -c https://huggingface.co/unsloth/FLUX.2-klein-9B-GGUF/resolve/main/flux-2-klein-9b-Q4_K_M.gguf -P ./models/diffusion_models/
!wget -c https://huggingface.co/Comfy-Org/Ideogram-4/resolve/main/vae/flux2-vae.safetensors -P ./models/vae/
!wget -c https://huggingface.co/Qwen/Qwen3-VL-8B-Instruct-GGUF/resolve/main/Qwen3VL-8B-Instruct-Q4_K_M.gguf -P ./models/text_encoders/
!mkdir -p user/default/workflows
!wget -O user/default/workflows/flux2_klein_ultimate_v2.1.json \
"https://raw.githubusercontent.com/arenaRNcloud/comfyui_workflows/main/workflows/FLUX.2%20Klein%209B%20%E2%80%94%20Ultimate%206-in-1%20Workflow%20(Face%20Swap%20%C2%B7%20Inpaint%20%C2%B7%20Auto-Mask%20%C2%B7%20NAG%20%C2%B7%20Refine%20%C2%B7%20Upscale)%20%E2%80%94%208GB%20VRAM/flux2_klein_ultimate_v2.1.json


## 3) Model Download OPTION B
Download the required model files (and optional components) into ComfyUI folders.


In [ ]:
# @title 3) Download models (FLUX.1 SRPO GGUF + quantized T5 GGUF + CLIP-L + official VAE)
import os

COMFY = '/content/ComfyUI'

UNET_DIR = f'{COMFY}/models/unet'
DIFF_DIR = f'{COMFY}/models/diffusion_models'
TE_DIR   = f'{COMFY}/models/text_encoders'
CLIP_DIR = f'{COMFY}/models/clip'
VAE_DIR  = f'{COMFY}/models/vae'

for d in [UNET_DIR, DIFF_DIR, TE_DIR, CLIP_DIR, VAE_DIR]:
    os.makedirs(d, exist_ok=True)


def dl(url, outdir, fname):
    outpath = os.path.join(outdir, fname)
    if os.path.exists(outpath):
        print('Already exists:', outpath)
        return
    print('Downloading:', fname)

    hf_token = os.environ.get('HUGGINGFACE_TOKEN') or os.environ.get('HF_TOKEN')
    if hf_token and 'huggingface.co' in url:
        !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M --header="Authorization: Bearer {hf_token}" "{url}" -d "{outdir}" -o "{fname}"
    else:
        !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "{url}" -d "{outdir}" -o "{fname}"


# SRPO GGUF
srpo_fname = f"srpo-{SRPO_QUANT}.gguf"
srpo_url = f"https://huggingface.co/befox/SRPO-GGUF/resolve/main/{srpo_fname}"
dl(srpo_url, UNET_DIR, srpo_fname)

srpo_link = f"{DIFF_DIR}/{srpo_fname}"
if not os.path.exists(srpo_link):
    !ln -s "{UNET_DIR}/{srpo_fname}" "{srpo_link}"

# Quantized T5-XXL GGUF
t5_fname = f"t5-v1_1-xxl-encoder-{T5_QUANT}.gguf"
t5_url = f"https://huggingface.co/city96/t5-v1_1-xxl-encoder-gguf/resolve/main/{t5_fname}"
dl(t5_url, TE_DIR, t5_fname)

t5_clip_link = f"{CLIP_DIR}/{t5_fname}"
if not os.path.exists(t5_clip_link):
    !ln -s "{TE_DIR}/{t5_fname}" "{t5_clip_link}"

# CLIP-L
clip_fname = 'clip_l.safetensors'
clip_url = 'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors'
dl(clip_url, CLIP_DIR, clip_fname)

clip_te_link = f"{TE_DIR}/{clip_fname}"
if not os.path.exists(clip_te_link):
    !ln -s "{CLIP_DIR}/{clip_fname}" "{clip_te_link}"

# Official FLUX VAE (token/license may be required)
vae_fname = 'ae.safetensors'
vae_url = 'https://huggingface.co/black-forest-labs/FLUX.1-schnell/resolve/main/ae.safetensors'
dl(vae_url, VAE_DIR, vae_fname)

print('Done downloading models.')
print('SRPO :', srpo_fname)
print('T5   :', t5_fname)
print('CLIP :', clip_fname)
print('VAE  :', vae_fname)


# ---- Lenovo UltraReal LoRA pack (Civitai) ----
DOWNLOAD_LENOVO_ULTRAREAL_LORAS = True
CIVITAI_API_TOKEN = os.environ.get('CIVITAI_API_TOKEN', '').strip()  # optional, helps if Civitai returns 403.
CURRENT_BASE_MODELS = ['Flux.1 D']

if DOWNLOAD_LENOVO_ULTRAREAL_LORAS:
    import json
    import subprocess
    import urllib.request
    import urllib.parse

    LORA_DIR = f'{COMFY}/models/loras'
    os.makedirs(LORA_DIR, exist_ok=True)

    def add_civitai_token(url, token):
        if not token:
            return url
        parts = urllib.parse.urlsplit(url)
        query = urllib.parse.parse_qsl(parts.query, keep_blank_values=True)
        if not any(k == 'token' for k, _ in query):
            query.append(('token', token))
        return urllib.parse.urlunsplit((parts.scheme, parts.netloc, parts.path, urllib.parse.urlencode(query), parts.fragment))

    def dl_civitai(url, outdir, fname, token=''):
        outpath = os.path.join(outdir, fname)
        if os.path.exists(outpath):
            print('Already exists:', outpath)
            return True

        final_url = add_civitai_token(url, token)
        print('Downloading LoRA:', fname)

        # 1) Fast path via aria2c
        cmd = ['aria2c', '--console-log-level=error', '-c', '-x', '8', '-s', '8', '-k', '1M', final_url, '-d', outdir, '-o', fname]
        rc = subprocess.run(cmd, check=False).returncode
        if rc == 0 and os.path.exists(outpath) and os.path.getsize(outpath) > 0:
            return True

        # 2) Fallback via urllib (some Civitai links fail in aria2 despite valid token)
        try:
            headers = {'User-Agent': 'Mozilla/5.0'}
            if token:
                headers['Authorization'] = f'Bearer {token}'
            req = urllib.request.Request(final_url, headers=headers)
            with urllib.request.urlopen(req, timeout=120) as resp, open(outpath, 'wb') as f:
                f.write(resp.read())
            if os.path.exists(outpath) and os.path.getsize(outpath) > 0:
                print('  Fallback downloader: OK')
                return True
        except Exception as e:
            print('  Fallback downloader failed:', e)

        if not token:
            print('  Download failed without token. Set CIVITAI_API_TOKEN env var and retry Cell 3.')
        else:
            print('  Download failed even with token. Check token validity and Civitai availability.')
        return False

    try:
        req = urllib.request.Request(
            'https://civitai.com/api/v1/models/1662740',
            headers={'User-Agent': 'Mozilla/5.0'}
        )
        with urllib.request.urlopen(req, timeout=30) as resp:
            civitai_model = json.loads(resp.read().decode('utf-8'))

        versions = civitai_model.get('modelVersions', [])
        print('\nLenovo UltraReal LoRA versions found:', len(versions))
        if not CURRENT_BASE_MODELS:
            print('Current notebook base-model tags: (none matched explicitly)')
        else:
            print('Current notebook base-model tags:', ', '.join(CURRENT_BASE_MODELS))

        for mv in versions:
            base = mv.get('baseModel', 'Unknown')
            match = base in CURRENT_BASE_MODELS
            mark = 'MATCH' if match else 'OTHER'
            files = [f for f in mv.get('files', []) if f.get('type') == 'Model']
            print(f"- [{mark}] {base} :: {mv.get('name', 'Unnamed version')} ({len(files)} file(s))")

        ok_count = 0
        total_count = 0
        matched_versions = [mv for mv in versions if mv.get('baseModel', 'Unknown') in CURRENT_BASE_MODELS]

        if not matched_versions:
            print('No MATCH versions found for this notebook base-model tag set. Skipping LoRA download.')

        for mv in matched_versions:
            for fobj in mv.get('files', []):
                if fobj.get('type') != 'Model':
                    continue
                total_count += 1
                raw_name = (fobj.get('name') or '').strip()
                base_tag = ''.join(ch if ch.isalnum() else '_' for ch in str(mv.get('baseModel', 'unknown'))).strip('_').lower() or 'unknown'
                fname = f"lenovo_ultrareal_{base_tag}_v{mv.get('id', 'x')}_f{fobj.get('id', 'x')}.safetensors"
                if raw_name and raw_name != fname:
                    print(f"  Civitai file name: {raw_name} -> saved as {fname}")
                if dl_civitai(fobj.get('downloadUrl', ''), LORA_DIR, fname, CIVITAI_API_TOKEN):
                    ok_count += 1

        print(f"\nLenovo UltraReal LoRA download summary (MATCH only): {ok_count}/{total_count} files ready in {LORA_DIR}")

    except Exception as e:
        print('Failed to fetch Lenovo UltraReal LoRA metadata from Civitai:', e)
        print('Tip: this may require CIVITAI_API_TOKEN or temporary Civitai availability.')


## 4) Download Check
Verify downloaded file sizes and calculate a rough VRAM estimate from real files.


In [ ]:
# @title 4) Verify downloads: actual file sizes and VRAM estimate
import os


def size_gb(path):
    return os.path.getsize(path) / (1024**3)


def est_vram_from_file(path, overhead=1.10):
    return size_gb(path) * overhead


srpo_fname = f"srpo-{SRPO_QUANT}.gguf"
t5_fname = f"t5-v1_1-xxl-encoder-{T5_QUANT}.gguf"

paths = [
    ('/content/ComfyUI/models/unet', srpo_fname),
    ('/content/ComfyUI/models/text_encoders', t5_fname),
    ('/content/ComfyUI/models/clip', 'clip_l.safetensors'),
    ('/content/ComfyUI/models/vae', 'ae.safetensors'),
]

print('Files:')
total_vram = 0.0
for d, f in paths:
    p = os.path.join(d, f)
    if not os.path.exists(p):
        print('MISSING:', p)
        continue
    s = size_gb(p)
    v = est_vram_from_file(p)
    total_vram += v
    print(f"- {p}\n  size~{s:.2f} GB  -> VRAM(weights)~{v:.2f} GB")

print(f"\nSum VRAM(weights) if all kept on GPU at once: ~{total_vram:.2f} GB")
print('Reminder: real peak VRAM will be higher (resolution/batch/latents/frames).')


## 5) Launch ComfyUI
Start ComfyUI and use the Cloudflare Tunnel URL shown in the output.


In [ ]:
# @title 5) Launch ComfyUI (Cloudflare Tunnel) - stable mode
import subprocess, threading, time, socket, os, re, sys, urllib.request

%cd /content/ComfyUI

# Install cloudflared (if not installed yet)
if not os.path.exists("cloudflared-linux-amd64.deb"):
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb


def wait_port(host="127.0.0.1", port=8188, timeout=240):
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            with socket.create_connection((host, port), timeout=1):
                return True
        except OSError:
            time.sleep(0.5)
    return False


def verify_tunnel_url(url, timeout=10):
    base = url.rstrip('/')
    candidates = [
        base + '/',
        base + '/api/system_stats',
        base + '/object_info',
    ]
    for c in candidates:
        try:
            req = urllib.request.Request(c, headers={'User-Agent': 'Mozilla/5.0'}, method='GET')
            with urllib.request.urlopen(req, timeout=timeout) as r:
                code = r.getcode()
                if 200 <= code < 500:
                    return True
        except Exception:
            pass
    return False


def wait_reachable_stable(url, proc, warmup_timeout=180, stable_successes=3, probe_interval=2.5):
    t0 = time.time()
    ok_streak = 0
    while time.time() - t0 < warmup_timeout:
        if proc.poll() is not None:
            return False
        if verify_tunnel_url(url, timeout=10):
            ok_streak += 1
            if ok_streak >= stable_successes:
                return True
        else:
            ok_streak = 0
        time.sleep(probe_interval)
    return False


def start_cloudflare_tunnel_once(port=8188, protocol='http2', read_timeout=150, warmup_timeout=180):
    print(f"\nStarting Cloudflare Quick Tunnel (protocol={protocol})...\n")
    sys.stdout.flush()

    cmd = [
        'cloudflared', 'tunnel',
        '--no-autoupdate',
        '--url', f'http://127.0.0.1:{port}',
        '--protocol', protocol,
        '--loglevel', 'info',
    ]

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    tunnel_patterns = [
        re.compile(r'https://[a-z0-9-]+\.trycloudflare\.com', re.I),
        re.compile(r'https://[a-z0-9-]+\.cfargotunnel\.com', re.I),
    ]
    ignore_patterns = [
        re.compile(r'https://www\.cloudflare\.com/website-terms/?', re.I),
    ]

    t0 = time.time()
    url = None

    while time.time() - t0 < read_timeout:
        line = proc.stdout.readline()
        if not line:
            if proc.poll() is not None:
                break
            time.sleep(0.1)
            continue

        s = line.strip()

        if any(ip.search(s) for ip in ignore_patterns):
            continue

        for pat in tunnel_patterns:
            m = pat.search(s)
            if m:
                url = m.group(0)
                break
        if url:
            break

    if not url:
        print('Failed to parse tunnel URL.')
        if proc.poll() is None:
            proc.terminate()
            try:
                proc.wait(timeout=5)
            except Exception:
                proc.kill()
        return None, None

    print(f'Found tunnel URL: {url}')
    print('Waiting for Cloudflare propagation and stable readiness...')

    if wait_reachable_stable(url, proc, warmup_timeout=warmup_timeout, stable_successes=3, probe_interval=2.5):
        print('\n--------------------------------------------------')
        print('YOUR LINK:', url)
        print('--------------------------------------------------\n')
        return proc, url

    print('Tunnel URL did not become stably reachable in time.')
    if proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=5)
        except Exception:
            proc.kill()
    return None, url


def start_tunnel_with_retries(port=8188, max_attempts=6):
    protocols = ['http2', 'quic']
    for attempt in range(1, max_attempts + 1):
        proto = protocols[(attempt - 1) % len(protocols)]
        print(f"\n== Tunnel attempt {attempt}/{max_attempts} ==")
        proc, url = start_cloudflare_tunnel_once(
            port=port,
            protocol=proto,
            read_timeout=150,
            warmup_timeout=180,
        )
        if proc is not None and url:
            return proc, url
        time.sleep(min(8, 2 + attempt))
    return None, None


def stop_proc_safely(proc):
    if proc is None:
        return
    if proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=5)
        except Exception:
            proc.kill()


def tunnel_thread(port=8188):
    if not wait_port('127.0.0.1', port, timeout=240):
        print('Timed out waiting for ComfyUI port', port)
        return

    print('\nComfyUI port is open. Creating stable tunnel...\n')

    while True:
        proc, url = start_tunnel_with_retries(port=port, max_attempts=6)
        if proc is None:
            print('Failed to establish a reachable Cloudflare tunnel automatically.')
            print('Rerun this cell to retry with a fresh tunnel session.')
            return

        unhealthy_streak = 0
        while proc.poll() is None:
            time.sleep(15)
            if verify_tunnel_url(url, timeout=8):
                unhealthy_streak = 0
            else:
                unhealthy_streak += 1
                print(f'Cloudflare tunnel health check failed ({unhealthy_streak}/3).')
                if unhealthy_streak >= 3:
                    print('Tunnel became unhealthy. Recreating...')
                    stop_proc_safely(proc)
                    break

        if proc.poll() is not None:
            rc = proc.returncode
            print(f'Cloudflare tunnel exited (code={rc}). Recreating...')


threading.Thread(target=tunnel_thread, daemon=True, args=(8188,)).start()

print('Starting ComfyUI... link will appear above.')
!python main.py --dont-print-server --port 8188 --lowvram --preview-method auto


/content/ComfyUI
Selecting previously unselected package cloudflared.
(Reading database ... 122477 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.7.0) ...
Setting up cloudflared (2026.7.0) ...
Processing triggers for man-db (2.10.2-1) ...
Starting ComfyUI... link will appear above.
[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[START] Security scan
[DONE] Security scan
## ComfyUI-Manager: installing dependencies done.
** ComfyUI startup time: 2026-07-09 10:14:00.090
** Platform: Linux
** Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
** Python executable: /usr/bin/python3
** ComfyUI Path: /content/ComfyUI
** ComfyUI Base Folder Path: /content/C

--2026-07-09 10:04:43--  https://huggingface.co/unsloth/FLUX.2-klein-9B-GGUF/resolve/main/flux-2-klein-9b-Q4_K_M.gguf
Resolving huggingface.co (huggingface.co)... 18.164.174.17, 18.164.174.118, 18.164.174.23, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.17|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.aws.cdn.hf.co/xet-bridge-us/696920ab190d7903fd610653/262df1a3a98bc328911e03d4d0f5d7e3b1397477a2c485b24509b6d115259aef?X-Xet-Cas-Uid=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27flux-2-klein-9b-Q4_K_M.gguf%3B+filename%3D%22flux-2-klein-9b-Q4_K_M.gguf%22%3B&user_id=public&Expires=1783595083&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5hd3MuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjk2OTIwYWIxOTBkNzkwM2ZkNjEwNjUzLzI2MmRmMWEzYTk4YmMzMjg5MTFlMDNkNGQwZjVkN2UzYjEzOTc0NzdhMmM0ODViMjQ1MDliNmQxMTUyNTlhZWZcXD9YLVhldC1DYXMtVWlkPXB1YmxpYyZyZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPWlubGluZSUzQitmaWxlbmFtZSUyQSUzRFVURi04